# 10 月交易明细清洗

本 Notebook 从原始工作簿独立筛选 10 月窗口，并沿用数据质量检查中的日期条件。

In [1]:
import pandas as pd
data = pd.read_excel('../data/raw/Online Retail.xlsx')

In [2]:
# 转换日期类型
data["InvoiceDate"] = pd.to_datetime(data["InvoiceDate"])

In [3]:
# 按数据质量检查使用的日期条件筛选 10 月交易
data_10_primary = data.loc[
    (data["InvoiceDate"] >= pd.Timestamp('2011-10-01')) & 
    (data["InvoiceDate"] < pd.Timestamp('2011-11-01'))
]
data_10 = data_10_primary.copy()
date_min = data_10['InvoiceDate'].min()
date_max = data_10['InvoiceDate'].max()
print(f"筛选后日期范围为{date_min}--{date_max}")
print(f"转换后缺失行为{data_10['InvoiceDate'].isna().sum()}")
print(f"转换后时间类型为{data_10['InvoiceDate'].dtypes}")
# 与数据质量检查的日期窗口结果一致

筛选后日期范围为2011-10-02 10:32:00--2011-10-31 17:19:00
转换后缺失行为0
转换后时间类型为datetime64[ns]


In [4]:
data_10.dtypes

InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object

In [5]:
# 列名规范化
data_10.columns = (
    data_10.columns
    .str.strip()
    .str.lower()
)
data_10.columns

Index(['invoiceno', 'stockcode', 'description', 'quantity', 'invoicedate',
       'unitprice', 'customerid', 'country'],
      dtype='object')

In [6]:
# 查看10月数据基本信息
x,y = data_10.shape
print(f"10月份数据为{x}行{y}列")
data_10.info()

10月份数据为60742行8列
<class 'pandas.core.frame.DataFrame'>
Index: 60742 entries, 370931 to 431672
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   invoiceno    60742 non-null  object        
 1   stockcode    60742 non-null  object        
 2   description  60638 non-null  object        
 3   quantity     60742 non-null  int64         
 4   invoicedate  60742 non-null  datetime64[ns]
 5   unitprice    60742 non-null  float64       
 6   customerid   50695 non-null  float64       
 7   country      60742 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 4.2+ MB


In [7]:
# 对完全重复的明细隔离整组，保留隔离记录供核验。整组隔离的实现参考了 AI 提示。
data_10_dup = data_10[data_10.duplicated(keep=False)]
data_10 = data_10.drop_duplicates(keep=False)
assert not data_10.duplicated(keep=False).any()

print(f"去重后数据为{len(data_10)}行")
print(f"重复数据为{len(data_10_dup)}")
# 隔离后 59,260 行，加上隔离的 1,482 行，与窗口内 60,742 行一致。

去重后数据为59260行
重复数据为1482


In [8]:
# 隔离重复组后，显式排除缺失发票编号的明细。
raw_missing_invoice_count = data_10_primary['InvoiceNo'].isna().sum()
missing_invoice = data_10['invoiceno'].isna()
excluded_missing_invoice_count = missing_invoice.sum()
data_10 = data_10.loc[~missing_invoice].copy()

print(f"原始 10 月窗口缺失发票号：{raw_missing_invoice_count:,} 行")
print(f"隔离重复组后因缺失发票号实际排除：{excluded_missing_invoice_count:,} 行")
print(f"排除缺失发票号后剩余：{len(data_10):,} 行")

原始 10 月窗口缺失发票号：0 行
隔离重复组后因缺失发票号实际排除：0 行
排除缺失发票号后剩余：59,260 行


In [9]:
# 发票编号以 C 或 c 开头表示取消。
cancel_mask = data_10['invoiceno'].astype("string").str.upper().str.startswith("C", na=False)
data_10_c = data_10[cancel_mask]
data_10 = data_10.drop(index=data_10_c.index)

print(f"排除后数据为{len(data_10)}行")
print(f"以c开头数据{len(data_10_c)}行")
# 58,072 行保留明细加上 1,188 行取消明细，与隔离后的 59,260 行一致。

排除后数据为58072行
以c开头数据1188行


In [10]:
# 仅保留数量和单价都为正的商品行。
positive_mask = (data_10['quantity'] > 0) & (data_10['unitprice'] > 0)
data_10_cdn = data_10[~positive_mask]
data_10 = data_10[positive_mask]
print(f"排除不符合条件后的数据为{len(data_10)}行")
print(f"不符合条件数据{len(data_10_cdn)}行")

排除不符合条件后的数据为57834行
不符合条件数据238行


In [11]:
# 添加新列 lineamount
data_10['lineamount'] = data_10['quantity'] * data_10['unitprice']
print(f"添加新列后数据为{len(data_10)}行")
# 数据为57834行，9列，无丢失
data_10['lineamount'].isna().sum()
# 新增列无缺失值

添加新列后数据为57834行


np.int64(0)

In [12]:
data_10.to_csv('../data/processed/data_10month.csv',index=False)

## 清洗对账

- 10 月窗口共 60,742 行。整组隔离重复明细 1,482 行后剩余 59,260 行。
- 对隔离后的数据显式排除缺失发票编号，再排除 C/c 开头的取消明细。本期缺失排除 0 行，取消排除 1,188 行，剩余 58,072 行。
- 排除 238 行非正数量或价格明细，最终保留 57,834 行。